In [1]:
# Source - https://stackoverflow.com/a
# Posted by G M, modified by community. See post 'Timeline' for change history
if 'google.colab' in str(get_ipython()):
  !git clone https://github.com/Vladislavicious/jenga_ml.git
  %cd jenga_ml
  !git switch dev

  !pip install -r requirements.txt
else:
  print('Not running on CoLab')

Not running on CoLab


In [2]:
import random
import os

from environment import make_jenga_env

import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.env_util import make_vec_env


In [3]:
n_blocks = 1
random.seed(123)
np.random.seed(123)
CHECK_STEPS = 1024

def make_env():
    env = make_jenga_env(n_blocks=n_blocks, render=True)
    return env


In [4]:
# env = make_env()

num_envs = 8
env = make_vec_env(make_env, n_envs=num_envs, vec_env_cls=SubprocVecEnv)
env = VecNormalize(
    env,
    norm_obs=True,
    norm_reward=False,
    clip_obs=10.0,
    training=True
)

In [5]:

model = PPO(
    "MlpPolicy",
    env,
    batch_size=128,
    verbose=1,
    seed=123,
)

model.learn(total_timesteps=300000)

Using cpu device
------------------------------
| time/              |       |
|    fps             | 1884  |
|    iterations      | 1     |
|    time_elapsed    | 8     |
|    total_timesteps | 16384 |
------------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 1396         |
|    iterations           | 2            |
|    time_elapsed         | 23           |
|    total_timesteps      | 32768        |
| train/                  |              |
|    approx_kl            | 0.0074009676 |
|    clip_fraction        | 0.0723       |
|    clip_range           | 0.2          |
|    entropy_loss         | -4.39        |
|    explained_variance   | 0.0871       |
|    learning_rate        | 0.0003       |
|    loss                 | 0.163        |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00366     |
|    value_loss           | 7.49         |
-----------------------------------

In [6]:
model.save("final_model")

In [7]:
single_env = make_env()

In [ ]:
obs, _ = single_env.reset()
for i in range(CHECK_STEPS * 2):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = single_env.step(action)
    single_env.render()
    if terminated or truncated:
        obs, _ = single_env.reset()
    single_env.env.debug_output()
    if i == CHECK_STEPS - 2:
        print("hi")

In [9]:
env.env.debug_output()

AttributeError: 'SubprocVecEnv' object has no attribute 'env'